# Provider Anomaly Detection (Isolation Forest)

This notebook performs anomaly detection on provider-level behavior using PostgreSQL as the source of truth.

## Deliverables
- Train Isolation Forest model
- Generate anomaly scores
- Identify high-risk providers
- Explain why each flagged provider was flagged
- Export scored and flagged outputs

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f"{x:,.4f}")

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5433')
DB_NAME = os.getenv('DB_NAME', 'medicare_provider_analytics')
DB_USER = os.getenv('DB_USER', 'medicare_user')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'medicare_password')

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


def run_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)


print(f"Connected target: {DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
provider_feature_sql = """
WITH provider_agg AS (
    SELECT
        p.provider_key,
        p.npi,
        p.last_org_name,
        p.provider_type,
        SUM(COALESCE(f.tot_srvcs, 0)) AS total_services,
        SUM(COALESCE(f.tot_benes, 0)) AS total_beneficiaries,
        SUM(COALESCE(f.avg_sbmtd_chrg, 0) * COALESCE(f.tot_srvcs, 0)) AS weighted_charge_sum,
        SUM(COALESCE(f.avg_mdcr_pymt_amt, 0) * COALESCE(f.tot_srvcs, 0)) AS weighted_payment_sum,
        COUNT(DISTINCT s.hcpcs_code) AS unique_procedures,
        COUNT(DISTINCT f.data_year) AS years_active
    FROM fact_provider_service f
    JOIN dim_provider p ON f.provider_key = p.provider_key
    JOIN dim_service s ON f.service_key = s.service_key
    GROUP BY p.provider_key, p.npi, p.last_org_name, p.provider_type
)
SELECT
    provider_key,
    npi,
    last_org_name,
    provider_type,
    total_services,
    total_beneficiaries,
    CASE WHEN total_services = 0 THEN NULL ELSE weighted_charge_sum / total_services END AS avg_charge,
    CASE WHEN total_services = 0 THEN NULL ELSE weighted_payment_sum / total_services END AS avg_payment,
    CASE WHEN weighted_payment_sum = 0 THEN NULL ELSE weighted_charge_sum / weighted_payment_sum END AS charge_payment_ratio,
    unique_procedures,
    years_active
FROM provider_agg
ORDER BY provider_key;
"""

df = run_sql(provider_feature_sql)

feature_cols = [
    'total_services',
    'total_beneficiaries',
    'avg_charge',
    'avg_payment',
    'charge_payment_ratio',
    'unique_procedures',
    'years_active',
]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
df[feature_cols] = df[feature_cols].fillna(0)

print(f"Providers loaded: {len(df):,}")
df.head()

In [ ]:
# Train Isolation Forest
X = df[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(
    n_estimators=300,
    contamination=0.03,
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_scaled)

# score_samples: lower = more anomalous
raw_score = iso.score_samples(X_scaled)
# decision_function: negative usually means anomaly
decision_score = iso.decision_function(X_scaled)
labels = iso.predict(X_scaled)  # -1 anomaly, 1 normal

# Build an interpretable risk score where higher = riskier
scaled_rank = pd.Series(raw_score).rank(pct=True)
risk_score = (1 - scaled_rank) * 100

df_scored = df.copy()
df_scored['anomaly_label'] = labels
df_scored['is_anomaly'] = (df_scored['anomaly_label'] == -1).astype(int)
df_scored['iforest_raw_score'] = raw_score
df_scored['iforest_decision_score'] = decision_score
df_scored['anomaly_risk_score'] = risk_score.round(4)

df_scored['risk_tier'] = pd.cut(
    df_scored['anomaly_risk_score'],
    bins=[-0.1, 80, 95, 100],
    labels=['Monitor', 'High', 'Critical'],
)

print('Anomaly label counts:')
print(df_scored['is_anomaly'].value_counts())
df_scored[['npi', 'provider_type', 'anomaly_risk_score', 'risk_tier', 'is_anomaly']].head()

In [ ]:
# Explain why providers were flagged using absolute z-score contributions per feature.
z_matrix = pd.DataFrame(
    np.abs(X_scaled),
    columns=[f"z_{c}" for c in feature_cols],
    index=df_scored.index,
)

df_explain = pd.concat([df_scored, z_matrix], axis=1)


def top_reasons(row: pd.Series, top_n: int = 3) -> str:
    reason_scores = {
        feature: row[f"z_{feature}"]
        for feature in feature_cols
    }
    ranked = sorted(reason_scores.items(), key=lambda kv: kv[1], reverse=True)[:top_n]
    return '; '.join([f"{k} (z={v:.2f})" for k, v in ranked])


df_explain['flag_reason_top3'] = df_explain.apply(top_reasons, axis=1)

# High-risk providers: anomalies OR extreme risk score
high_risk_mask = (df_explain['is_anomaly'] == 1) | (df_explain['anomaly_risk_score'] >= 95)
df_high_risk = (
    df_explain.loc[high_risk_mask, [
        'provider_key',
        'npi',
        'last_org_name',
        'provider_type',
        'anomaly_risk_score',
        'risk_tier',
        'is_anomaly',
        'total_services',
        'total_beneficiaries',
        'avg_charge',
        'avg_payment',
        'charge_payment_ratio',
        'unique_procedures',
        'years_active',
        'flag_reason_top3',
    ]]
    .sort_values(['anomaly_risk_score', 'is_anomaly'], ascending=[False, False])
    .reset_index(drop=True)
)

print(f"High-risk providers identified: {len(df_high_risk):,}")
df_high_risk.head(25)

In [ ]:
# Quick diagnostics and distributions by risk tier
risk_summary = (
    df_explain.groupby('risk_tier', observed=False)
    .agg(
        providers=('provider_key', 'count'),
        avg_risk_score=('anomaly_risk_score', 'mean'),
        anomaly_rate=('is_anomaly', 'mean'),
        avg_total_services=('total_services', 'mean'),
        avg_avg_payment=('avg_payment', 'mean'),
    )
    .reset_index()
)

risk_summary['anomaly_rate'] = (risk_summary['anomaly_rate'] * 100).round(2)
risk_summary

In [ ]:
# Export scored outputs
output_dir = Path.cwd().parent / 'Data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

all_scores_path = output_dir / 'provider_anomaly_scores.csv'
high_risk_path = output_dir / 'high_risk_providers.csv'

export_cols = [
    'provider_key',
    'npi',
    'last_org_name',
    'provider_type',
    'total_services',
    'total_beneficiaries',
    'avg_charge',
    'avg_payment',
    'charge_payment_ratio',
    'unique_procedures',
    'years_active',
    'iforest_raw_score',
    'iforest_decision_score',
    'anomaly_risk_score',
    'risk_tier',
    'is_anomaly',
    'flag_reason_top3',
]

df_explain[export_cols].to_csv(all_scores_path, index=False)
df_high_risk.to_csv(high_risk_path, index=False)

print(f"Exported: {all_scores_path}")
print(f"Exported: {high_risk_path}")
print(f"Rows exported (all scored): {len(df_explain):,}")
print(f"Rows exported (high risk): {len(df_high_risk):,}")